In [ ]:
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 100

import shap
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
import torch

from sklearn.decomposition import PCA
import networkx as nx
from torch_geometric.utils import to_networkx

from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.loader import DataLoader

from torch_geometric.explain.config import ModelConfig
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error


np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data():
    """
    Loads (160,4) data from 'Feature_guide_target_baseonly_PAM_complete_HTCas9_complete.txt'.
    """
    data_branch1 = []
    current_array = []
    with open('Feature_guide_target_baseonly_PAM_complete_HTCas9_complete.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 160, 4)
    return X_branch1

def load_branch2_data():
    """
    Loads (280,3) data from 'Feature_guide_target_complete_struct_only_bhah_HTCas9.txt'.
    """
    data_branch2 = []
    current_array = []
    with open('Feature_guide_target_complete_struct_only_bhah_HTCas9.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch2.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch2.append(current_array)
    X_branch2 = np.array(data_branch2)
    X_branch2 = X_branch2.reshape(len(X_branch2), 280, 3)
    return X_branch2

def load_branch3_data(filename="Feature_guide_target_complete_value_features_17D_HTCas9.txt"):
    """
    Loads 17-dimensional numeric features from file.
    Each line has 17 floats (space- or comma-delimited).
    """
    X_branch3 = []
    with open(filename, 'r') as file:
        for line in file:
            line = line.strip()
            # Expect 17 numbers per line
            values = line.replace('[', '').replace(']', '').split()
            float_vals = [float(v.strip().replace(',', '')) for v in values]
            X_branch3.append(float_vals)
    
    X_branch3 = np.array(X_branch3)  # shape: (n_samples, 17)
    X_branch3 = X_branch3.reshape(len(X_branch3), 17)
    print("X_branch3 shape:", X_branch3.shape)
    return X_branch3
    
def load_reaction_rates():
    with open('HTCas9_indel_frequency_value_percentage.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)

########################################
# 2. Graph Data Utilities (for GNN)
########################################

def nucleotide_to_one_hot(nucleotide):
    mapping = {
        'A': [1, 0, 0, 0],
        'C': [0, 1, 0, 0],
        'G': [0, 0, 1, 0],
        'T': [0, 0, 0, 1],
        'U': [0, 0, 0, 1],
        '-': [0, 0, 0, 0]
    }
    return mapping.get(nucleotide, [0,0,0,0])

def encode_sequence(sequence):
    return [nucleotide_to_one_hot(nuc) for nuc in sequence]

def matrix_to_edge_index(prob_matrix):
    edges_source = []
    edges_target = []
    L = len(prob_matrix)
    for i in range(L):
        for j in range(i + 1, L):  # only i < j
            if prob_matrix[i][j] > 0.01:
                edges_source.append(i)
                edges_target.append(j)
    return [edges_source, edges_target]

def generate_edge_features(edge_index, seq_one_hot, prob_matrix):
    features = []
    if not edge_index:
        return []
    for i in range(len(edge_index[0])):
        src_idx = edge_index[0][i]
        tgt_idx = edge_index[1][i]
        prob_val = prob_matrix[src_idx][tgt_idx]
        
        def decode_nt(one_hot_vec):
            if one_hot_vec[0] == 1:
                return 'A'
            elif one_hot_vec[1] == 1:
                return 'C'
            elif one_hot_vec[2] == 1:
                return 'G'
            elif one_hot_vec[3] == 1:
                return 'T'
            return 'N'
        
        src_char = decode_nt(seq_one_hot[src_idx])
        tgt_char = decode_nt(seq_one_hot[tgt_idx])
        
        if (src_char == 'A' and tgt_char in ['T','U']) or (src_char in ['T','U'] and tgt_char == 'A'):
            pair_type = 2
        elif (src_char == 'G' and tgt_char in ['T','U']) or (src_char in ['T','U'] and tgt_char == 'G'):
            pair_type = 2
        elif (src_char == 'C' and tgt_char == 'G') or (src_char == 'G' and tgt_char == 'C'):
            pair_type = 3
        
        features.append([pair_type/3, prob_val])
    return features

# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def RNA_reverse_complement(RNA):
    complement = {'A': 'U', 'C': 'G', 'G': 'C', 'U': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(RNA))

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def load_dna_reaction_data():
    with open('HTCas9_full_guide_sequences.txt', 'r') as f:
        crna_sequences = [line.strip() for line in f.readlines()]
    with open('HTCas9_target_sequences_noPAM.txt', 'r') as f:
        target_sequences = [line.strip() for line in f.readlines()]
    
    crna_node_features = [encode_sequence(seq) for seq in crna_sequences]
    
    target_bh_sequences = [target_sequences[i] + DNA_reverse_complement(target_sequences[i]) for i in range(len(target_sequences))]
    target_bh_node_features = [encode_sequence(seq) for seq in target_bh_sequences]
    
    duplex_sequences = [crna_sequences[i] + target_sequences[i] for i in range(len(target_sequences))]
    duplex_node_features = [encode_sequence(seq) for seq in duplex_sequences]

    target_ah_node_features = [encode_sequence(DNA_reverse_complement(seq)) for seq in target_sequences]

    crna_edge_indices = []
    crna_edge_features = []
    for guide in crna_sequences:
        prob_matrix = pairs(strands=guide, model=my_model_RNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_guide = [nucleotide_to_one_hot(n) for n in guide]
        feats = generate_edge_features(edge_index, oh_guide, prob_matrix)
        crna_edge_indices.append(edge_index)
        crna_edge_features.append(feats)

    target_bh_edge_indices = []
    target_bh_edge_features = []
    for i in range (0, len(crna_sequences)):
        prob_matrix = pairs(strands=[target_sequences[i], DNA_reverse_complement(target_sequences[i])], model=my_model_DNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_duplex = [nucleotide_to_one_hot(n) for n in target_bh_sequences[i]]
        feats = generate_edge_features(edge_index, oh_duplex, prob_matrix)
        target_bh_edge_indices.append(edge_index)
        target_bh_edge_features.append(feats)

    
    duplex_edge_indices = []
    duplex_edge_features = []
    for i in range (0, len(crna_sequences)):
        prob_matrix = pairs(strands=[crna_sequences[i], target_sequences[i]], model=my_model_DNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_duplex = [nucleotide_to_one_hot(n) for n in duplex_sequences[i]]
        feats = generate_edge_features(edge_index, oh_duplex, prob_matrix)
        duplex_edge_indices.append(edge_index)
        duplex_edge_features.append(feats)

    target_ah_edge_indices = []
    target_ah_edge_features = [] 
    for target in target_sequences:
        prob_matrix = pairs(strands=DNA_reverse_complement(target), model=my_model_DNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_guide = [nucleotide_to_one_hot(n) for n in DNA_reverse_complement(target)]
        feats = generate_edge_features(edge_index, oh_guide, prob_matrix)
        target_ah_edge_indices.append(edge_index)
        target_ah_edge_features.append(feats)
    
    with open('HTCas9_indel_frequency_value_percentage.txt', 'r') as f:
        reaction_rates = [float(line.strip()) for line in f.readlines()]
    
    return (crna_node_features, target_bh_node_features, duplex_node_features, target_ah_node_features,
            crna_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
            crna_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_features,
            reaction_rates)


class CRNADuplexSubstructuresDataset(Dataset):
    def __init__(self, crna_node_features, target_bh_node_features, duplex_node_features, target_ah_node_features,
                 crna_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
                 crna_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_features,
                 reaction_rates):
        super().__init__()
        self.crna_node_features = crna_node_features
        self.target_bh_node_features = target_bh_node_features
        self.duplex_node_features = duplex_node_features
        self.target_ah_node_features = target_ah_node_features
        self.crna_edge_indices = crna_edge_indices
        self.target_bh_edge_indices = target_bh_edge_indices
        self.duplex_edge_indices = duplex_edge_indices
        self.target_ah_edge_indices = target_ah_edge_indices
        self.crna_edge_features = crna_edge_features
        self.target_bh_edge_features = target_bh_edge_features
        self.duplex_edge_features = duplex_edge_features
        self.target_ah_edge_features = target_ah_edge_features
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples
    def __getitem__(self, idx):
        c_x = torch.tensor(self.crna_node_features[idx], dtype=torch.float)
        c_edge_index_list = self.crna_edge_indices[idx]
        if not c_edge_index_list:
            c_edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            c_edge_index = torch.tensor(c_edge_index_list, dtype=torch.long)
        c_edge_attr = torch.tensor(self.crna_edge_features[idx], dtype=torch.float)
        if c_edge_attr.dim() == 1:
            c_edge_attr = c_edge_attr.unsqueeze(-1).repeat(1,2)
        cRNA_data = Data(x=c_x, edge_index=c_edge_index, edge_attr=c_edge_attr)

        t_bh_x = torch.tensor(self.target_bh_node_features[idx], dtype=torch.float)
        t_bh_edge_index_list = self.target_bh_edge_indices[idx]
        if not t_bh_edge_index_list:
            t_bh_edge_index = torch.empty((2,0), dtype=torch.long)

        else:
            t_bh_edge_index = torch.tensor(t_bh_edge_index_list, dtype=torch.long)
        t_bh_edge_attr = torch.tensor(self.target_bh_edge_features[idx], dtype=torch.float)
        if t_bh_edge_attr.dim() == 1:
            t_bh_edge_attr = t_bh_edge_attr.unsqueeze(-1).repeat(1,2)
        target_bh_data = Data(x=t_bh_x, edge_index=t_bh_edge_index, edge_attr=t_bh_edge_attr)
        
        d_x = torch.tensor(self.duplex_node_features[idx], dtype=torch.float)
        d_edge_index_list = self.duplex_edge_indices[idx]
        if not d_edge_index_list:
            d_edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            d_edge_index = torch.tensor(d_edge_index_list, dtype=torch.long)
        d_edge_attr = torch.tensor(self.duplex_edge_features[idx], dtype=torch.float)
        if d_edge_attr.dim() == 1:
            d_edge_attr = d_edge_attr.unsqueeze(-1).repeat(1,2)
        duplex_data = Data(x=d_x, edge_index=d_edge_index, edge_attr=d_edge_attr)

        t_ah_x = torch.tensor(self.target_ah_node_features[idx], dtype=torch.float)
        t_ah_edge_index_list = self.target_ah_edge_indices[idx]
        if not t_ah_edge_index_list:
            t_ah_edge_index = torch.empty((2,0), dtype=torch.long)

        else:
            t_ah_edge_index = torch.tensor(t_ah_edge_index_list, dtype=torch.long)
        t_ah_edge_attr = torch.tensor(self.target_ah_edge_features[idx], dtype=torch.float)
        if t_ah_edge_attr.dim() == 1:
            t_ah_edge_attr = t_ah_edge_attr.unsqueeze(-1).repeat(1,2)
        target_ah_data = Data(x=t_ah_x, edge_index=t_ah_edge_index, edge_attr=t_ah_edge_attr)
        
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return cRNA_data, target_bh_data, duplex_data, target_ah_data, y_val


def graph_collate(batch):
    from torch_geometric.data import Batch
    cRNA_list, target_bh_list, duplex_list, target_ah_list, y_list = zip(*batch)
    batch_cRNA = Batch.from_data_list(cRNA_list)
    batch_target_bh = Batch.from_data_list(target_bh_list)
    batch_duplex = Batch.from_data_list(duplex_list)
    batch_target_ah = Batch.from_data_list(target_ah_list)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return batch_cRNA, batch_target_bh, batch_duplex, batch_target_ah, y


########################################
# 3. CNN, GNN, MLP Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (160,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*80, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,80)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,80)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


class CNNBranch2(nn.Module):
    """
    CNN branch for (280,3).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(3, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*140, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,3,280)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,140)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import NNConv, global_mean_pool, global_max_pool


class GNNBranch(nn.Module):
    def __init__(self, hidden_dim=32, edge_attr_dim=2):
        super().__init__()
        self.hidden_dim = hidden_dim

        def make_edge_mlp(edge_attr_dim, in_channels, out_channels):
            return nn.Sequential(
                nn.Linear(edge_attr_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, in_channels * out_channels)
            )

        # cRNA
        self.edge_mlp_c = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_c2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_c = NNConv(4, hidden_dim, self.edge_mlp_c, aggr='mean')
        self.conv2_c = NNConv(hidden_dim, hidden_dim, self.edge_mlp_c2, aggr='mean')

        # target_bh
        self.edge_mlp_bh = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_bh2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_t_bh = NNConv(4, hidden_dim, self.edge_mlp_bh, aggr='mean')
        self.conv2_t_bh = NNConv(hidden_dim, hidden_dim, self.edge_mlp_bh2, aggr='mean')

        # duplex
        self.edge_mlp_d = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_d2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_d = NNConv(4, hidden_dim, self.edge_mlp_d, aggr='mean')
        self.conv2_d = NNConv(hidden_dim, hidden_dim, self.edge_mlp_d2, aggr='mean')

        # target_ah
        self.edge_mlp_ah = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_ah2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_t_ah = NNConv(4, hidden_dim, self.edge_mlp_ah, aggr='mean')
        self.conv2_t_ah = NNConv(hidden_dim, hidden_dim, self.edge_mlp_ah2, aggr='mean')

        # MLP after pooling all graph embeddings
        self.mlp_merge = nn.Sequential(
            nn.Linear(hidden_dim * 4 * 2, hidden_dim),  # 4 graphs, mean+max
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def pool(self, x, batch):
        return torch.cat([
            global_mean_pool(x, batch),
            global_max_pool(x, batch)
        ], dim=1)


    def forward(self, crna_data, target_bh_data, duplex_data, target_ah_data):
        def process(graph_data, conv1, conv2):
            x, edge_index, edge_attr, batch = (
                graph_data.x, graph_data.edge_index, graph_data.edge_attr, graph_data.batch
            )
            x = F.elu(conv1(x, edge_index, edge_attr))
            x = F.elu(conv2(x, edge_index, edge_attr))
            return self.pool(x, batch)

        x_c = process(crna_data, self.conv1_c, self.conv2_c)
        x_t_bh = process(target_bh_data, self.conv1_t_bh, self.conv2_t_bh)
        x_d = process(duplex_data, self.conv1_d, self.conv2_d)
        x_t_ah = process(target_ah_data, self.conv1_t_ah, self.conv2_t_ah)

        merged = torch.cat([x_c, x_t_bh, x_d, x_t_ah], dim=1)
        return self.mlp_merge(merged)



class MLPBranch17(nn.Module):
    """
    A single hidden layer MLP for 17D => output dimension mlp_dim.
    No separate "output" layer; just one fc + ReLU => final embedding.
    """
    def __init__(self, input_dim=17, hidden_dim=32):
        super().__init__()
        self.fc = nn.Linear(input_dim, hidden_dim)
    def forward(self, x):
        # x shape: (batch,17)
        x = F.relu(self.fc(x))  # (batch,hidden_dim)
        return x
        

########################################
# 4. Final Fusion Model
########################################


class CNN_GNN_MLP_Fusion(nn.Module):
    """
    End-to-end: 
      - CNNBranch1 => feat1
      - CNNBranch2 => feat2
      - GNNBranch  => feat_gnn
      - MLPBranch17 => feat_mlp
    Concat => dropout => final FC => 1
    """
    def __init__(self,
                 filters1, kernel_size1, dense_units1,  # CNN1
                 filters2, kernel_size2, dense_units2,  # CNN2
                 gnn_hidden_dim,
                 mlp_hidden_dim,  # single hidden dimension for MLP
                 final_fc_dim,
                 dropout_rate=0.0):  # new hyperparameter for dropout
        super().__init__()
        
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.cnn_branch2 = CNNBranch2(filters2, kernel_size2, dense_units2)
        self.gnn_branch = GNNBranch(hidden_dim=gnn_hidden_dim)
        self.mlp_branch = MLPBranch17(input_dim=17, hidden_dim=mlp_hidden_dim)
        
        # total dimension = (dense_units1 + dense_units2 + gnn_hidden_dim + mlp_hidden_dim)
        total_dim = dense_units1 + dense_units2 + gnn_hidden_dim + mlp_hidden_dim
        
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(total_dim, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)


    
    def forward(self, crna_data, target_bh_data, duplex_data, target_ah_data, x1, x2, x3):
        feat1 = self.cnn_branch1(x1)                   # (batch, dense_units1)
        feat2 = self.cnn_branch2(x2)                   # (batch, dense_units2)
        feat_gnn = self.gnn_branch(crna_data, target_bh_data, duplex_data, target_ah_data)  # (batch, gnn_hidden_dim)
        feat_mlp = self.mlp_branch(x3)                 # (batch, mlp_hidden_dim)
        
        merged = torch.cat([feat1, feat2, feat_gnn, feat_mlp], dim=1)
        # Apply dropout on the concatenated features
        merged = F.dropout(merged, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))   # (batch, final_fc_dim)
        out = self.out(x)                 # (batch, 1)
        return out.view(-1)


########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, graph_dataset, X1, X2, X3, reaction_rates):
        super().__init__()
        self.graph_dataset = graph_dataset
        self.X1 = X1
        self.X2 = X2
        self.X3 = X3
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
        
        assert len(graph_dataset) == self.num_samples
        assert len(X1) == self.num_samples
        assert len(X2) == self.num_samples
        assert len(X3) == self.num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        crna_data, target_bh_data, duplex_data, target_ah_data, y_val = self.graph_dataset[idx]
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)  
        x2 = torch.tensor(self.X2[idx], dtype=torch.float)  
        x3 = torch.tensor(self.X3[idx], dtype=torch.float)  
        return crna_data, target_bh_data, duplex_data, target_ah_data, x1, x2, x3, y_val
        

def hybrid_collate(batch):
    from torch_geometric.data import Batch
    cRNA_list, target_bh_list, duplex_list, target_ah_list, x1_list, x2_list, x3_list, y_list = zip(*batch)
    batch_cRNA = Batch.from_data_list(cRNA_list)
    batch_target_bh = Batch.from_data_list(target_bh_list)
    batch_duplex = Batch.from_data_list(duplex_list)
    batch_target_ah = Batch.from_data_list(target_ah_list)
    x1 = torch.stack(x1_list, dim=0)
    x2 = torch.stack(x2_list, dim=0)
    x3 = torch.stack(x3_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return batch_cRNA, batch_target_bh, batch_duplex, batch_target_ah, x1, x2, x3, y


class FullModelWrapperForCNN2(nn.Module):
    def __init__(self, full_model, 
                 fixed_x1, fixed_x3,
                 fixed_crna_data, fixed_target_bh_data, fixed_duplex_data, fixed_target_ah_data):
        super().__init__()
        self.full_model = full_model.eval()
        self.fixed_x1 = fixed_x1
        self.fixed_x3 = fixed_x3
        self.fixed_crna_data = fixed_crna_data
        self.fixed_target_bh_data = fixed_target_bh_data
        self.fixed_duplex_data = fixed_duplex_data
        self.fixed_target_ah_data = fixed_target_ah_data

    def forward(self, x2):
        batch_size = x2.shape[0]

        # Repeat fixed inputs to match batch size
        x1 = self.fixed_x1.expand(batch_size, -1, -1)
        x3 = self.fixed_x3.expand(batch_size, -1)

        # Repeat graph data (use PyG's Batch for proper batching)
        from torch_geometric.data import Batch
        batch_crna = Batch.from_data_list([self.fixed_crna_data] * batch_size)
        batch_target_bh = Batch.from_data_list([self.fixed_target_bh_data] * batch_size)
        batch_duplex = Batch.from_data_list([self.fixed_duplex_data] * batch_size)
        batch_target_ah = Batch.from_data_list([self.fixed_target_ah_data] * batch_size)

        out = self.full_model(
            batch_crna, batch_target_bh, batch_duplex, batch_target_ah,
            x1, x2, x3
        )
        return out.view(-1, 1)  # Return shape: (batch, 1)



def filter_extreme_by_column(matrix, threshold=0.2):
    """
    For each column in the matrix:
    - If the range (max - min) exceeds the threshold,
    - Keep only the max and min values.
    - Set other values to 0.
    """
    filtered = np.zeros_like(matrix)
    for col in range(matrix.shape[1]):
        col_vals = matrix[:, col]
        col_max = np.max(col_vals)
        col_min = np.min(col_vals)
        if (col_max - col_min) >= threshold:
            max_idx = np.argmax(col_vals)
            min_idx = np.argmin(col_vals)
            filtered[max_idx, col] = col_max
            filtered[min_idx, col] = col_min
    return filtered


import torch
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd


# Segment ranges: (start, end)
segments_cnn1 = [(0, 100), (100, 160)]
segments_cnn2 = [(0, 100), (100, 140), (140, 240), (240, 280)]


np.random.seed(42)
torch.manual_seed(42)


# === Example Execution ===
if __name__ == "__main__":
    # Load your trained model
    full_model = torch.load(
    "trial_32_model_single_hidden_mlp_complete_HTCas9_L2_DROPOUT_complete.pt",
    weights_only=False
)
    full_model.eval()

    from sklearn.cluster import KMeans
    from sklearn.metrics import pairwise_distances_argmin_min
    import torch
    import numpy as np
    
    # Load original data
    X1_all = load_branch1_data()
    X2_all = load_branch2_data()
    X3_all = load_branch3_data()
    rates = load_reaction_rates()
    
    # Flatten for clustering
    X1_flat = X1_all.reshape(X1_all.shape[0], -1)
    X2_flat = X2_all.reshape(X2_all.shape[0], -1)
    X3_flat = X3_all  # already flat (N, 24)
    
    # Run KMeans
    kmeans1_sample = KMeans(n_clusters=1000, random_state=42).fit(X1_flat)
    kmeans2_sample = KMeans(n_clusters=1000, random_state=42).fit(X2_flat)
    kmeans3_sample = KMeans(n_clusters=1000, random_state=42).fit(X3_flat)
    kmeans1_background = KMeans(n_clusters=100, random_state=42).fit(X1_flat)
    kmeans2_background = KMeans(n_clusters=100, random_state=42).fit(X2_flat)
    kmeans3_background = KMeans(n_clusters=100, random_state=42).fit(X3_flat)
    
    # Get indices of closest original points to each centroid
    closest_X1_indices_sample, _ = pairwise_distances_argmin_min(kmeans1_sample.cluster_centers_, X1_flat)
    closest_X2_indices_sample, _ = pairwise_distances_argmin_min(kmeans2_sample.cluster_centers_, X2_flat)
    closest_X3_indices_sample, _ = pairwise_distances_argmin_min(kmeans3_sample.cluster_centers_, X3_flat)
    closest_X1_indices_background, _ = pairwise_distances_argmin_min(kmeans1_background.cluster_centers_, X1_flat)
    closest_X2_indices_background, _ = pairwise_distances_argmin_min(kmeans2_background.cluster_centers_, X2_flat)
    closest_X3_indices_background, _ = pairwise_distances_argmin_min(kmeans3_background.cluster_centers_, X3_flat)

    # Get samples 

    X1_selected = X1_all[closest_X1_indices_sample]
    X2_selected = X2_all[closest_X2_indices_sample]
    X3_selected = X3_all[closest_X3_indices_sample]


    # Get backgrounds
    background_cnn1 = X1_all[closest_X1_indices_background]
    background_cnn2 = X2_all[closest_X2_indices_background]
    background_mlp = X3_all[closest_X3_indices_background]
    
    # Use these indices to extract the corresponding reaction rates

    rates_X1 = rates[closest_X1_indices_sample]
    rates_X2 = rates[closest_X2_indices_sample]
    rates_X3 = rates[closest_X3_indices_sample]

    

    (_, _, _, _,
     crna_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
     crna_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_features,
     reaction_rates) = load_dna_reaction_data()
    crna_node_features, target_bh_node_features, duplex_node_features, target_ah_node_features = load_dna_reaction_data()[:4]
    graph_dataset = CRNADuplexSubstructuresDataset(
        crna_node_features, target_bh_node_features, duplex_node_features, target_ah_node_features,
        crna_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
        crna_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_features,
        reaction_rates
    )

    # Fixed samples in different branches (sample is the one with median reaction rate)
    sample_idx = np.argsort(rates)[len(rates) // 2]  # index of median value
    x1_sample = torch.tensor(X1_all[sample_idx], dtype=torch.float).unsqueeze(0)  
    x2_sample = torch.tensor(X2_all[sample_idx], dtype=torch.float).unsqueeze(0)  
    x3_sample = torch.tensor(X3_all[sample_idx], dtype=torch.float).unsqueeze(0) 
    crna, target_bh, duplex, target_ah, _ = graph_dataset[sample_idx]
    
    

    # === CNN Branch 2 SHAP ===
    
    # 1. Create the wrapper
    wrapper = FullModelWrapperForCNN2(
        full_model=full_model,
        fixed_x1=x1_sample,
        fixed_x3=x3_sample,
        fixed_crna_data=crna,
        fixed_target_bh_data=target_bh,
        fixed_duplex_data=duplex,
        fixed_target_ah_data=target_ah
    )
    
    # 2. Build background and input tensors
    background_cnn2 = torch.tensor(background_cnn2, dtype=torch.float) 
    X2_tensor = torch.tensor(X2_selected, dtype=torch.float)    
    X2_all_tensor = torch.tensor(X2_all, dtype=torch.float)   
    
    
    # 3. Run SHAP
    explainer = shap.GradientExplainer(wrapper, background_cnn2)
    shap_values_cnn2 = explainer.shap_values(X2_tensor)
    print("SHAP shape:", shap_values_cnn2[0].shape)

    wrapper.eval()
    
    shap_array = shap_values_cnn2
    shap_array = shap_array.squeeze(-1)  
    avg_shap = shap_array.mean(axis=0) 

    heatmap_data = avg_shap.T 
    global_vmin = heatmap_data.min()
    global_vmax = heatmap_data.max()

    filtered_heatmap_data = filter_extreme_by_column(heatmap_data, threshold=global_vmax*0.1)


    positions = np.arange(1, 281)
    states = ["Unpaired", "Paired-Opening", "Paired-Closing"]
    feature_names_cnn2 = [f"Pos{pos-260}_{state}" for pos in positions for state in states]
    
    X2_flat = X2_tensor.view(X2_tensor.size(0), -1).numpy()
    shap_values_cnn2_flat = shap_values_cnn2.squeeze().reshape(X2_tensor.size(0), -1)
    
    
    segments_cnn2_ssDNA_target_ah = [(260, 280)]
    for seg_start, seg_end in segments_cnn2_ssDNA_target_ah:
        seg_feature_indices = list(range(seg_start * 3, seg_end * 3))  # 3 features per position
        seg_feature_names = [feature_names_cnn2[i] for i in seg_feature_indices]
        
        import pandas as pd
        X_values = X2_flat[:, seg_feature_indices].cpu().numpy() if hasattr(X2_flat[:, seg_feature_indices], "cpu") else X2_flat[:, seg_feature_indices]
        
        df_input = pd.DataFrame(X_values, columns=seg_feature_names)
        df_shap = pd.DataFrame( shap_values_cnn2_flat[:, seg_feature_indices], columns=seg_feature_names)
        
        df_combined = pd.concat([df_input, df_shap], axis=1)

        df_combined.to_csv("shap_values_cnn2_Cas9_ssDNA_target_ah_32.csv", index=False)

print('finished')


In [ ]:
# Models tested: 32, 43, 62, 63, 65, 67, 70, 71, 74, 82

In [ ]:
import pandas as pd
import csv
from scipy.stats import spearmanr
from collections import Counter

# --- Config ---
file_names = [
    "shap_values_cnn2_Cas9_ssDNA_target_ah_32.csv", "shap_values_cnn2_Cas9_ssDNA_target_ah_43.csv",
    "shap_values_cnn2_Cas9_ssDNA_target_ah_62.csv", "shap_values_cnn2_Cas9_ssDNA_target_ah_63.csv",
    "shap_values_cnn2_Cas9_ssDNA_target_ah_65.csv", "shap_values_cnn2_Cas9_ssDNA_target_ah_67.csv",
    "shap_values_cnn2_Cas9_ssDNA_target_ah_70.csv", "shap_values_cnn2_Cas9_ssDNA_target_ah_71.csv",
    "shap_values_cnn2_Cas9_ssDNA_target_ah_74.csv", "shap_values_cnn2_Cas9_ssDNA_target_ah_82.csv"
]

results = []

# Step 1: collect results
for file_name in file_names:
    model_id = file_name.split("_")[-1].replace(".csv", "")  # Extract model ID like '27'

    with open(file_name) as f:
        reader = csv.reader(f)
        header = next(reader)

    df = pd.read_csv(file_name, header=None, skiprows=1)
    df.columns = header  # overwrite with true header

    col_counts = Counter(df.columns)
    duplicate_features = [col for col, count in col_counts.items() if count > 1]

    for feature in duplicate_features:
        indices = [i for i, col in enumerate(df.columns) if col == feature]
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):
                col1 = df.iloc[:, indices[i]]
                col2 = df.iloc[:, indices[j]]
                corr, pval = spearmanr(col1, col2)
                results.append({
                    "Feature": feature,
                    "Model": model_id,
                    "Spearman Correlation": corr,
                    "P-Value": pval
                })

# Step 2: Build DataFrame and order features
df = pd.DataFrame(results)

# Track first appearance order of features
feature_order = []
seen = set()
for row in results:
    feat = row["Feature"]
    if feat not in seen:
        seen.add(feat)
        feature_order.append(feat)

# Use category sorting to preserve order
df["Feature_order"] = pd.Categorical(df["Feature"], categories=feature_order, ordered=True)
df = df.sort_values(by=["Feature_order", "Model"])
df = df.drop(columns=["Feature_order"])

# Step 3: Blank out repeated feature names
def clear_repeated_features(df):
    cleared = []
    current_feat = None
    for _, row in df.iterrows():
        feat = row["Feature"]
        if feat == current_feat:
            row["Feature"] = ""
        else:
            current_feat = feat
        cleared.append(row)
    return pd.DataFrame(cleared)

df_clean = clear_repeated_features(df)

# Step 4: Save CSV
df_clean.to_csv("spearman_duplicates_grouped_norepeat_SpCas9_cnn2_ssDNA_target_ah.csv", index=False)
print("Saved with feature names grouped and de-duplicated to 'spearman_duplicates_grouped_norepeat_SpCas9_cnn2_ssDNA_target_ah.csv'")
